# Model tuning — Modelado-Pneumonia (Google Colab)

Clones the repository, installs dependencies into an isolated virtual environment, and runs `scripts/tune_models.py` / `scripts/compare_models.py` against the shared remote database (`results_unsa_ira`) — the same one used from a local machine. No local state to lose: each search persists progress to the database as it runs.

**Resumable by design**: if a Colab session disconnects mid-search, rerunning the same cell with the same parameters picks up where it left off. Any trial already persisted is detected and reused instantly (never recomputed); only the trial in flight at the moment of disconnection needs to rerun.

**Isolated environment**: project dependencies are installed into `repo/venv`, not into Colab's system Python. Colab's own stack (`ipykernel`, `google.colab`, `tensorflow-text`, etc.) ships pinned versions of packages this project also depends on (pandas, numpy, tensorflow); installing directly on top of it produces dependency-resolver conflicts that can destabilize the notebook kernel itself. A separate venv avoids that entirely — `google.colab.userdata` still runs fine in the notebook's own kernel, and every project script runs through the venv's Python instead.

## 1. Clone the repository and create an isolated virtual environment

In [ ]:
REPO_URL = "https://github.com/<org>/Modelado-Pneumonia.git"
BRANCH = "feature/modelling-basics"

!git clone --branch "$BRANCH" "$REPO_URL" repo
%cd repo
# Colab's system Python often ships an incomplete `ensurepip`, which makes the
# standard library's `python -m venv` fail to bootstrap pip inside the new
# environment. The `virtualenv` package bundles its own pip bootstrap and
# doesn't hit that issue.
!pip install -q virtualenv
!virtualenv -q venv
!./venv/bin/pip install -q --upgrade pip
!./venv/bin/pip install -q -r requirements.txt

## 2. Database credentials

Store the connection string as a Colab Secret (key icon 🔑 in the left panel) named `DATABASE_URL`, with the same value used in the project's local `.env` file, and grant this notebook access to it. Do not paste credentials directly into a cell.

This runs in the notebook's own kernel (where `google.colab` is available), not inside `venv` — the resulting environment variable is still inherited by every `venv/bin/python` subprocess invoked from later cells.

In [ ]:
import os
from google.colab import userdata

os.environ["DATABASE_URL"] = userdata.get("DATABASE_URL")

# Optional: override the project-wide random seed for this session (default is
# already 42). Only needed for a deliberate seed-robustness check.
# os.environ["RANDOM_SEED"] = "42"

## 3. Prepare the data (one-time per Colab instance)

`data/` is not part of the git repository (raw and processed data files are gitignored), so a fresh clone has none of it. This downloads the raw table from PostgreSQL and rebuilds `data/processed/iras_weekly_clean.csv`, which every model/script reads from — without it, `get_departmental_data()` fails with `FileNotFoundError: Data file not found: data/raw/iras_data_raw.csv`.

Only needs to run once per Colab instance (the files persist for the rest of the session); rerun with `--force` if the source table changed and the cached processed file needs rebuilding.

In [ ]:
!./venv/bin/python scripts/prepare_data.py

## 4. Sanity check (database connectivity + project import)

Runs through `venv/bin/python` so it exercises the exact same environment the tuning scripts will use.

In [ ]:
!./venv/bin/python -c "from pneumonia.data.load_data import get_db_engine; from sqlalchemy import text; eng = get_db_engine(None); conn = eng.connect(); n = conn.execute(text('SELECT COUNT(*) FROM results_unsa_ira.walkforward_runs')).scalar(); print(f'Connection OK -- {n} runs already stored in the shared database.')"

## 5. GPU check (only relevant for LSTM/GRU)

SARIMA, RandomForest, XGBoost, Prophet, HoltWinters, and the Naive baselines never use a GPU — running them under a GPU runtime wastes Colab's limited GPU quota for no benefit. Only LSTM/GRU (Keras/TensorFlow) actually accelerate on one. Switch runtime type (Runtime → Change runtime type) accordingly before tuning.

In [ ]:
!./venv/bin/python -c "import tensorflow as tf; gpus = tf.config.list_physical_devices('GPU'); print('GPU available:', bool(gpus), gpus if gpus else '(CPU runtime -- fine for everything except LSTM/GRU)')"

## 6. Tune one model/department

Edit the parameters below and run the cell. If the session disconnects partway through, rerunning this same cell resumes automatically.

In [ ]:
DEPARTMENT = "AMAZONAS"
MODEL = "SARIMA"          # SARIMA, RandomForest, XGBoost, Prophet, HoltWinters, LSTM, GRU
SEARCH_METHOD = "optuna"  # optuna (recommended for larger search spaces), grid (small spaces, e.g. HoltWinters), random
N_ITER = 30                # ignored when SEARCH_METHOD=grid
AGE_GROUP = "under5"       # under5 | 60plus

!./venv/bin/python scripts/tune_models.py \
    --department "$DEPARTMENT" \
    --model "$MODEL" \
    --age_group "$AGE_GROUP" \
    --search_method "$SEARCH_METHOD" \
    --n_iter "$N_ITER"

## 7. Batch run across departments/models

Each (department, model) combination is independently resumable — if the session is interrupted, rerunning this cell skips whatever already completed (detected almost instantly, no recomputation) and continues with what remains.

In [ ]:
import subprocess

VENV_PYTHON = "./venv/bin/python"
DEPARTMENTS = ["AMAZONAS", "LIMA", "CUSCO"]
MODELS = ["SARIMA", "XGBoost"]
SEARCH_METHOD = "optuna"
N_ITER = 30
AGE_GROUP = "under5"

failed = []
for dept in DEPARTMENTS:
    for model in MODELS:
        print(f"\n{'='*70}\n{dept} / {model}\n{'='*70}")
        result = subprocess.run([
            VENV_PYTHON, "scripts/tune_models.py",
            "--department", dept, "--model", model, "--age_group", AGE_GROUP,
            "--search_method", SEARCH_METHOD, "--n_iter", str(N_ITER),
        ])
        if result.returncode != 0:
            failed.append((dept, model))

if failed:
    print(f"\nFailed: {failed}")
else:
    print("\nAll combinations completed without errors.")

## 8. View results (protected holdout number, by default)

In [ ]:
!./venv/bin/python scripts/compare_models.py --department "$DEPARTMENT" --source db